# Single-Feature Deep Dive: same_month_workday_avg_qty_mean

Goal: analyze only `same_month_workday_avg_qty_mean`, focusing on its relationship with monthly total `actual_month_total` and its behavior across forecast anchors from `WD1` to `WD20+`.

Evaluation scope: this notebook is a **non-cross-sectional time-series feature review**. There is only one monthly aggregate target per `bizym`, so true cross-sectional IC is intentionally excluded. Pearson/Spearman metrics below are time-series association checks, not cross-sectional factor IC.

Constraint: do not introduce other predictive features, multi-feature models, SHAP, or feature-importance analysis. `bizym`, `actual_month_total`, `month_total_workdays`, and `forecast_workday_seq` are used only as index, label, grouping, or scale-restoration fields.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

try:
    import statsmodels.api as sm
    from statsmodels.stats.diagnostic import acorr_ljungbox
    from statsmodels.stats.stattools import durbin_watson
    STATSMODELS_AVAILABLE = True
except Exception:
    sm = None
    acorr_ljungbox = None
    durbin_watson = None
    STATSMODELS_AVAILABLE = False

warnings.filterwarnings("ignore", category=UserWarning)

FEATURE_COL = "same_month_workday_avg_qty_mean"
TARGET_COL = "actual_month_total"
HISTORY_YEARS = (1, 2, 3)
ROLLING_WINDOW_MONTHS = 12
MIN_ROLLING_PERIODS = 6
HOLDOUT_FRACTION = 0.30
MIN_WALK_FORWARD_TRAIN_MONTHS = 24

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.unicode_minus"] = False


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "sales_daily.csv").exists():
            return candidate
    raise FileNotFoundError("Cannot locate repo root containing data/sales_daily.csv")


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "sales_daily.csv"
print(f"repo root: {REPO_ROOT}")
print(f"data path: {DATA_PATH}")
print(f"statsmodels_available = {STATSMODELS_AVAILABLE}")


## 1. Data Loading and Workday Definition

The workday definition follows `03-experiment-20260701.ipynb`: use `chinese_calendar` when available, otherwise fall back to Monday-Friday.

In [ ]:
def build_is_workday_func():
    try:
        from chinese_calendar import is_workday as cn_is_workday

        return lambda ts: bool(cn_is_workday(pd.Timestamp(ts).date())), "chinese_calendar"
    except Exception:
        return lambda ts: pd.Timestamp(ts).dayofweek < 5, "weekday_fallback_mon_fri"

is_workday_func, calendar_source = build_is_workday_func()

daily_df = pd.read_csv(DATA_PATH, parse_dates=["transdate"])
daily_df = daily_df.sort_values(["bizym", "transdate"]).reset_index(drop=True)
daily_df["is_workday"] = daily_df["transdate"].map(is_workday_func).astype(bool)
daily_df["month_period"] = daily_df["transdate"].dt.to_period("M")

print(f"calendar_source = {calendar_source}")
print(f"rows = {len(daily_df):,}, months = {daily_df['bizym'].nunique():,}")
print(f"date range = {daily_df['transdate'].min().date()} ~ {daily_df['transdate'].max().date()}")
display(daily_df.head())

## 2. Single Feature Reproduction

In `03-experiment-20260701.ipynb`, the feature is defined as:

`same_month_workday_avg_qty_mean = mean(same_month_y1_workday_avg_qty, same_month_y2_workday_avg_qty, same_month_y3_workday_avg_qty)`

Each historical same-month workday average is `hist_month_total_qty / hist_month_workdays`.

In [ ]:
def add_month_index_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["year"] = out["bizym"] // 100
    out["month_num"] = out["bizym"] % 100
    out["month_start"] = pd.to_datetime(out["year"].astype(str) + "-" + out["month_num"].astype(str) + "-01")
    return out

monthly_base = (
    daily_df.groupby("bizym", as_index=False)
    .agg(
        actual_month_total=("qty", "sum"),
        month_total_workdays=("is_workday", "sum"),
        month_days=("transdate", "nunique"),
        first_date=("transdate", "min"),
        last_date=("transdate", "max"),
    )
)
monthly_base = add_month_index_fields(monthly_base)
monthly_base["hist_month_workday_avg_qty"] = (
    monthly_base["actual_month_total"] / monthly_base["month_total_workdays"].replace(0, np.nan)
)

feature_panel = monthly_base[[
    "bizym", "year", "month_num", "month_start", "actual_month_total", "month_total_workdays", "month_days"
]].copy()

for years_back in HISTORY_YEARS:
    lag_monthly = monthly_base[["bizym", "hist_month_workday_avg_qty"]].copy()
    lag_monthly["bizym"] = lag_monthly["bizym"] + years_back * 100
    lag_monthly = lag_monthly.rename(
        columns={"hist_month_workday_avg_qty": f"same_month_y{years_back}_workday_avg_qty"}
    )
    feature_panel = feature_panel.merge(lag_monthly, on="bizym", how="left")

history_cols = [f"same_month_y{y}_workday_avg_qty" for y in HISTORY_YEARS]
feature_panel["history_years_available"] = feature_panel[history_cols].notna().sum(axis=1)
feature_panel[FEATURE_COL] = feature_panel[history_cols].mean(axis=1)
feature_panel["single_feature_implied_month_total"] = (
    feature_panel[FEATURE_COL] * feature_panel["month_total_workdays"]
)
feature_panel["single_feature_error"] = (
    feature_panel["single_feature_implied_month_total"] - feature_panel[TARGET_COL]
)
feature_panel["single_feature_error_pct"] = feature_panel["single_feature_error"] / feature_panel[TARGET_COL] * 100
feature_panel["single_feature_mape_pct"] = feature_panel["single_feature_error_pct"].abs()

analysis_feature_cols = [FEATURE_COL]
assert analysis_feature_cols == ["same_month_workday_avg_qty_mean"]

display(feature_panel.head(12))
display(feature_panel.tail(12))

In [ ]:
coverage_summary = (
    feature_panel.groupby("history_years_available", as_index=False)
    .agg(months=("bizym", "count"), first_bizym=("bizym", "min"), last_bizym=("bizym", "max"))
)
missing_feature_months = feature_panel.loc[feature_panel[FEATURE_COL].isna(), ["bizym", "month_start", "history_years_available"]]

print("Historical same-month coverage:")
display(coverage_summary)
print("Months with missing feature values:")
display(missing_feature_months)

## 2.1 Non-Cross-Sectional Evaluation Contract

Because this dataset is a single monthly aggregate series, the right feature checks are time-series checks:

- Coverage and point-in-time availability: missing rate, historical years available, no current-month target leakage.
- Association: Pearson and Spearman correlation across months. These are time-series association metrics, not cross-sectional IC.
- Statistical significance: one-variable OLS coefficient t-stat and p-value, with HAC/Newey-West robust statistics when `statsmodels` is available.
- Predictive error: MAE, RMSE, MAPE, sMAPE, WAPE, bias, and tail error after restoring the feature to monthly-total scale.
- Temporal robustness: chronological holdout, expanding walk-forward OOS fit, rolling/expanding correlation, and year/month/WD slices.

Excluded by design: cross-sectional Spearman/Pearson IC, ICIR by cross-section, category/SKU/store slices, and multi-feature LightGBM importance. Those require a panel grain such as `bizym × sku` or belong in the full-model feature analysis.


In [ ]:
feature_coverage = pd.DataFrame([
    {
        "metric": "total_months",
        "value": len(feature_panel),
    },
    {
        "metric": "usable_months_with_feature_and_target",
        "value": feature_panel[[FEATURE_COL, TARGET_COL]].dropna().shape[0],
    },
    {
        "metric": "feature_missing_months",
        "value": feature_panel[FEATURE_COL].isna().sum(),
    },
    {
        "metric": "feature_missing_rate_pct",
        "value": feature_panel[FEATURE_COL].isna().mean() * 100,
    },
    {
        "metric": "full_3y_history_months",
        "value": (feature_panel["history_years_available"] == len(HISTORY_YEARS)).sum(),
    },
    {
        "metric": "full_3y_history_rate_pct",
        "value": (feature_panel["history_years_available"] == len(HISTORY_YEARS)).mean() * 100,
    },
    {
        "metric": "first_usable_bizym",
        "value": feature_panel.loc[feature_panel[FEATURE_COL].notna(), "bizym"].min(),
    },
    {
        "metric": "last_usable_bizym",
        "value": feature_panel.loc[feature_panel[FEATURE_COL].notna(), "bizym"].max(),
    },
])

display(feature_coverage)

leakage_contract = pd.DataFrame([
    {
        "check": "feature_source",
        "status": "pass",
        "detail": "Feature uses same-month historical workday averages from y1/y2/y3 only.",
    },
    {
        "check": "available_before_wd1",
        "status": "pass",
        "detail": "Feature does not depend on current-month observed sales and is fixed across WD anchors.",
    },
    {
        "check": "cross_sectional_ic_applicable",
        "status": "not_applicable",
        "detail": "Only one aggregate target per month; no same-time multiple entities to rank.",
    },
])
display(leakage_contract)


## 2.2 Universal Point-in-Time Leakage Audit

The leakage guard below is intentionally generic. It is not tied to `same_month_workday_avg_qty_mean`; it audits any time-series feature list through a point-in-time contract:

- every predictive feature must have a timing contract or fail closed;
- every feature contract must declare how to compute the latest raw `source_end_time` used by that feature;
- `source_end_time` must be earlier than, or explicitly allowed to equal, the prediction cutoff time;
- target, actual, prediction, error, and evaluation columns are blocked globally by exact name and regex patterns;
- family-specific recomputation checks are optional add-ons, not the core leakage guard.

For the current single-feature notebook, the prediction cutoff is `month_start` because this historical same-month feature should be available before `WD1`. For the full 300+ feature panel in `03-experiment-20260701.ipynb`, the same audit function should be called with `prediction_time_col="forecast_date"` or the relevant forecast cutoff column, and each feature family should register its source-end rule.


In [ ]:
@dataclass(frozen=True)
class FeatureTimingContract:
    feature_name: str
    family: str
    prediction_time_col: str
    source_end_func: Callable[[pd.DataFrame, dict], pd.Series] | None = None
    source_end_col: str | None = None
    allow_equal_prediction_time: bool = False
    availability_type: str = "historical_observation"
    required_columns: tuple[str, ...] = ()
    component_columns: tuple[str, ...] = ()
    recompute_func: Callable[[pd.DataFrame, dict], pd.Series] | None = None
    notes: str = ""


FORBIDDEN_FEATURE_EXACT_NAMES = {
    TARGET_COL,
    "target_daily_qty",
    "actual_month_total",
    "month_total_qty",
    "month_total_num_hosp",
    "single_feature_implied_month_total",
    "single_feature_error",
    "single_feature_error_pct",
    "single_feature_mape_pct",
    "linear_fit_month_total",
    "linear_fit_error",
    "linear_fit_mape_pct",
    "raw_predicted_month_total",
    "nonnegative_predicted_month_total",
    "month_total_mape",
    "split",
    "bizym",
    "month_start",
    "anchor_date",
    "forecast_date",
    "target_date",
}

FORBIDDEN_FEATURE_REGEXES = [
    re.compile(pattern, flags=re.IGNORECASE)
    for pattern in [
        r"(^|_)(actual|label|truth)($|_)",
        r"(^|_)(pred|prediction|predicted)($|_)",
        r"(^|_)(error|ape|mape|smape|wape|bias|residual)($|_)",
        r"(^|_)target_(daily_)?(qty|amount|sales|total)($|_)",
        r"(^|_)future_.*(qty|amount|sales|total)",
        r"(^|_)(lead|next)_.*(qty|amount|sales|total)",
        r"shift\s*\(-",
    ]
]

POINT_IN_TIME_CONTRACT_CATALOG = pd.DataFrame([
    {
        "feature_family": "lag / rolling history",
        "examples": "qty_lag_7d, qty_roll_14d_mean, *_through_anchor",
        "generic_source_end_rule": "latest raw observation used by the lag/window <= anchor_date or forecast cutoff",
        "audit_requirement": "contract computes per-row source_end_time from anchor/cutoff and window definition",
    },
    {
        "feature_family": "month-to-date observed",
        "examples": "mtd_qty, mtd_workday_avg_qty, anchor_day_qty",
        "generic_source_end_rule": "latest included observation <= anchor_date; for WDk 00:00 this must be <= previous completed workday",
        "audit_requirement": "contract declares whether anchor day is observable at prediction time",
    },
    {
        "feature_family": "historical same-month",
        "examples": "same_month_y1_total_qty, same_month_workday_avg_qty_mean",
        "generic_source_end_rule": "latest historical source month end < target month prediction cutoff",
        "audit_requirement": "contract maps target row to prior-year source rows and compares source month end to cutoff",
    },
    {
        "feature_family": "calendar / deterministic ex ante",
        "examples": "target_day_of_week, target_is_month_end, lunar holiday flags",
        "generic_source_end_rule": "deterministic calendar known at or before forecast creation; may equal prediction cutoff",
        "audit_requirement": "contract marks availability_type=deterministic_calendar and allow_equal_prediction_time=True",
    },
    {
        "feature_family": "external release / exogenous",
        "examples": "weather forecast, published macro, external inventory snapshot",
        "generic_source_end_rule": "vendor release timestamp <= prediction cutoff, not event timestamp alone",
        "audit_requirement": "contract must use release_time, ingestion_time, or effective_available_time",
    },
])

display(POINT_IN_TIME_CONTRACT_CATALOG)


def blocked_feature_reason(feature_name: str) -> str | None:
    if feature_name in FORBIDDEN_FEATURE_EXACT_NAMES:
        return "exact forbidden label/evaluation/id column"
    for regex in FORBIDDEN_FEATURE_REGEXES:
        if regex.search(feature_name):
            return f"matches forbidden regex: {regex.pattern}"
    return None


def same_month_history_source_end(df: pd.DataFrame, context: dict) -> pd.Series:
    monthly_lookup = context["monthly_base"].set_index("bizym")
    source_end_values = []
    for _, row in df.iterrows():
        target_bizym = int(row["bizym"])
        source_dates = []
        for years_back in HISTORY_YEARS:
            component_col = f"same_month_y{years_back}_workday_avg_qty"
            source_bizym = target_bizym - years_back * 100
            if source_bizym in monthly_lookup.index and pd.notna(row.get(component_col, np.nan)):
                source_dates.append(monthly_lookup.loc[source_bizym, "last_date"])
        source_end_values.append(max(source_dates) if source_dates else pd.NaT)
    return pd.Series(source_end_values, index=df.index, name="source_end_time")


def same_month_workday_avg_mean_recompute(df: pd.DataFrame, context: dict) -> pd.Series:
    return df[list(context["history_cols"])].mean(axis=1)


def build_same_month_component_detail(df: pd.DataFrame, monthly_base: pd.DataFrame) -> pd.DataFrame:
    monthly_lookup = monthly_base.set_index("bizym")
    rows = []
    for _, row in df.iterrows():
        target_bizym = int(row["bizym"])
        target_month_start = row["month_start"]
        for years_back in HISTORY_YEARS:
            component_col = f"same_month_y{years_back}_workday_avg_qty"
            source_bizym = target_bizym - years_back * 100
            source_exists = source_bizym in monthly_lookup.index
            component_value = row[component_col]
            if source_exists:
                source = monthly_lookup.loc[source_bizym]
                expected_component = source["actual_month_total"] / source["month_total_workdays"]
                source_last_date = source["last_date"]
                source_month_start = source["month_start"]
            else:
                expected_component = np.nan
                source_last_date = pd.NaT
                source_month_start = pd.NaT

            component_matches = (
                pd.isna(component_value) and pd.isna(expected_component)
            ) or np.isclose(component_value, expected_component, rtol=1e-10, atol=1e-8, equal_nan=True)

            rows.append({
                "target_bizym": target_bizym,
                "target_month_start": target_month_start,
                "years_back": years_back,
                "component_col": component_col,
                "source_bizym": source_bizym,
                "source_exists": source_exists,
                "source_month_start": source_month_start,
                "source_last_date": source_last_date,
                "source_bizym_before_target": source_bizym < target_bizym,
                "source_month_ends_before_target_month": bool(source_last_date < target_month_start) if source_exists else True,
                "component_is_missing_when_source_missing": bool(pd.isna(component_value)) if not source_exists else True,
                "component_matches_recomputed_history": bool(component_matches),
                "component_value": component_value,
                "expected_component": expected_component,
            })
    return pd.DataFrame(rows)


analysis_feature_contracts = {
    FEATURE_COL: FeatureTimingContract(
        feature_name=FEATURE_COL,
        family="historical_same_month_workday_average",
        prediction_time_col="month_start",
        source_end_func=same_month_history_source_end,
        allow_equal_prediction_time=False,
        availability_type="historical_observation",
        required_columns=("bizym", "month_start", *history_cols),
        component_columns=tuple(history_cols),
        recompute_func=same_month_workday_avg_mean_recompute,
        notes="Uses prior-year same-month full-month history; must be available before WD1.",
    )
}


def audit_point_in_time_features(
    data: pd.DataFrame,
    feature_cols: list[str],
    contracts: dict[str, FeatureTimingContract],
    context: dict,
    strict_unknown_features: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    detail_rows = []

    for feature_name in feature_cols:
        blocked_reason = blocked_feature_reason(feature_name)
        contract = contracts.get(feature_name)
        feature_exists = feature_name in data.columns
        missing_contract = contract is None
        missing_required_columns: list[str] = []
        time_violation_count = np.nan
        null_source_with_value_count = np.nan
        recompute_mismatch_count = np.nan
        contract_status = "pass"
        contract_detail = []

        if blocked_reason is not None:
            contract_status = "fail"
            contract_detail.append(blocked_reason)
        if not feature_exists:
            contract_status = "fail"
            contract_detail.append("feature column is missing from data")
        if missing_contract:
            if strict_unknown_features:
                contract_status = "fail"
                contract_detail.append("missing FeatureTimingContract; strict mode blocks unknown features")
            else:
                contract_status = "review_required"
                contract_detail.append("missing FeatureTimingContract; manual review required")

        if contract is not None and feature_exists:
            required = set(contract.required_columns) | {contract.prediction_time_col, feature_name}
            missing_required_columns = sorted(col for col in required if col not in data.columns)
            if missing_required_columns:
                contract_status = "fail"
                contract_detail.append("missing required columns: " + ", ".join(missing_required_columns))
            else:
                prediction_time = pd.to_datetime(data[contract.prediction_time_col])
                if contract.source_end_func is not None:
                    source_end_time = pd.to_datetime(contract.source_end_func(data, context))
                elif contract.source_end_col is not None:
                    source_end_time = pd.to_datetime(data[contract.source_end_col])
                elif contract.availability_type == "deterministic_calendar":
                    source_end_time = prediction_time
                else:
                    contract_status = "fail"
                    contract_detail.append("contract has no source_end_func/source_end_col")
                    source_end_time = pd.Series(pd.NaT, index=data.index)

                feature_has_value = data[feature_name].notna()
                source_missing = source_end_time.isna()
                if contract.allow_equal_prediction_time:
                    time_ok = source_end_time.le(prediction_time)
                    comparison_rule = "source_end_time <= prediction_time"
                else:
                    time_ok = source_end_time.lt(prediction_time)
                    comparison_rule = "source_end_time < prediction_time"

                value_missing_with_no_source = source_missing & ~feature_has_value
                time_pass_mask = time_ok | value_missing_with_no_source
                time_violation = ~time_pass_mask
                time_violation_count = int(time_violation.sum())
                null_source_with_value_count = int((source_missing & feature_has_value).sum())

                if time_violation_count > 0:
                    contract_status = "fail"
                    contract_detail.append(f"{time_violation_count} rows violate {comparison_rule}")
                if null_source_with_value_count > 0:
                    contract_status = "fail"
                    contract_detail.append(f"{null_source_with_value_count} rows have feature values without source_end_time")

                if contract.recompute_func is not None:
                    expected = contract.recompute_func(data, context)
                    actual = data[feature_name]
                    recompute_ok = np.isclose(actual, expected, rtol=1e-10, atol=1e-8, equal_nan=True)
                    recompute_mismatch_count = int((~pd.Series(recompute_ok, index=data.index)).sum())
                    if recompute_mismatch_count > 0:
                        contract_status = "fail"
                        contract_detail.append(f"{recompute_mismatch_count} rows mismatch recomputed feature values")

                sample = pd.DataFrame({
                    "feature_name": feature_name,
                    "row_index": data.index,
                    "prediction_time": prediction_time,
                    "source_end_time": source_end_time,
                    "feature_has_value": feature_has_value,
                    "time_ok": time_ok,
                    "time_violation": time_violation,
                    "comparison_rule": comparison_rule,
                })
                detail_rows.append(sample)

        summary_rows.append({
            "feature_name": feature_name,
            "family": contract.family if contract is not None else "unknown",
            "availability_type": contract.availability_type if contract is not None else "unknown",
            "status": contract_status,
            "blocked_reason": blocked_reason,
            "has_contract": contract is not None,
            "feature_exists": feature_exists,
            "missing_required_columns": ", ".join(missing_required_columns),
            "time_violation_rows": time_violation_count,
            "null_source_with_value_rows": null_source_with_value_count,
            "recompute_mismatch_rows": recompute_mismatch_count,
            "allow_equal_prediction_time": contract.allow_equal_prediction_time if contract is not None else np.nan,
            "prediction_time_col": contract.prediction_time_col if contract is not None else "",
            "detail": "; ".join(contract_detail) if contract_detail else "point-in-time contract passed",
        })

    summary = pd.DataFrame(summary_rows)
    detail = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()
    return summary, detail


def summarize_audit_checks(generic_summary: pd.DataFrame, same_month_detail: pd.DataFrame) -> pd.DataFrame:
    feature_failures = int(generic_summary["status"].ne("pass").sum())
    component_checks = pd.DataFrame([
        {
            "check": "feature_timing_contract_present_for_all_features",
            "scope": "universal",
            "status": "pass" if generic_summary["has_contract"].all() else "fail",
            "failed_rows": int((~generic_summary["has_contract"]).sum()),
            "detail": "Every feature must have a FeatureTimingContract in strict mode.",
        },
        {
            "check": "no_forbidden_label_or_evaluation_columns",
            "scope": "universal",
            "status": "pass" if generic_summary["blocked_reason"].isna().all() else "fail",
            "failed_rows": int(generic_summary["blocked_reason"].notna().sum()),
            "detail": "Feature names are screened against exact and regex forbidden label/evaluation/future patterns.",
        },
        {
            "check": "source_end_time_not_after_prediction_time",
            "scope": "universal",
            "status": "pass" if generic_summary["time_violation_rows"].fillna(0).sum() == 0 else "fail",
            "failed_rows": int(generic_summary["time_violation_rows"].fillna(0).sum()),
            "detail": "For each feature row, latest raw source timestamp must satisfy the feature contract cutoff rule.",
        },
        {
            "check": "no_value_without_source_end_time",
            "scope": "universal",
            "status": "pass" if generic_summary["null_source_with_value_rows"].fillna(0).sum() == 0 else "fail",
            "failed_rows": int(generic_summary["null_source_with_value_rows"].fillna(0).sum()),
            "detail": "Rows with missing source_end_time must not contain a non-null feature value.",
        },
        {
            "check": "registered_recompute_rules_match",
            "scope": "universal_optional",
            "status": "pass" if generic_summary["recompute_mismatch_rows"].fillna(0).sum() == 0 else "fail",
            "failed_rows": int(generic_summary["recompute_mismatch_rows"].fillna(0).sum()),
            "detail": "When a contract provides a recompute rule, feature values must match the recomputed values.",
        },
        {
            "check": "point_in_time_feature_audit_passed",
            "scope": "universal",
            "status": "pass" if feature_failures == 0 else "fail",
            "failed_rows": feature_failures,
            "detail": "Overall fail-closed feature timing audit result.",
        },
        {
            "check": "same_month_source_bizym_strictly_before_target",
            "scope": "family_specific_historical_same_month",
            "status": "pass" if same_month_detail["source_bizym_before_target"].all() else "fail",
            "failed_rows": int((~same_month_detail["source_bizym_before_target"]).sum()),
            "detail": "Same-month y1/y2/y3 source months are earlier than the target month.",
        },
        {
            "check": "same_month_source_month_ends_before_target_month",
            "scope": "family_specific_historical_same_month",
            "status": "pass" if same_month_detail["source_month_ends_before_target_month"].all() else "fail",
            "failed_rows": int((~same_month_detail["source_month_ends_before_target_month"]).sum()),
            "detail": "Historical source month last_date is before target month_start.",
        },
        {
            "check": "same_month_component_matches_recomputed_history",
            "scope": "family_specific_historical_same_month",
            "status": "pass" if same_month_detail["component_matches_recomputed_history"].all() else "fail",
            "failed_rows": int((~same_month_detail["component_matches_recomputed_history"]).sum()),
            "detail": "Same-month component values match historical actual_month_total / historical workdays.",
        },
    ])
    return component_checks


point_in_time_context = {
    "monthly_base": monthly_base,
    "history_cols": tuple(history_cols),
}

generic_feature_audit, point_in_time_detail = audit_point_in_time_features(
    data=feature_panel,
    feature_cols=analysis_feature_cols,
    contracts=analysis_feature_contracts,
    context=point_in_time_context,
    strict_unknown_features=True,
)
same_month_component_detail = build_same_month_component_detail(feature_panel, monthly_base)
leakage_audit = summarize_audit_checks(generic_feature_audit, same_month_component_detail)
leakage_detail = point_in_time_detail

print("Registered feature timing contracts:")
display(pd.DataFrame([
    {
        "feature_name": contract.feature_name,
        "family": contract.family,
        "availability_type": contract.availability_type,
        "prediction_time_col": contract.prediction_time_col,
        "allow_equal_prediction_time": contract.allow_equal_prediction_time,
        "component_columns": ", ".join(contract.component_columns),
        "notes": contract.notes,
    }
    for contract in analysis_feature_contracts.values()
]))

print("Generic point-in-time feature audit:")
display(generic_feature_audit)
print("Leakage audit checks:")
display(leakage_audit)

failed_leakage_checks = leakage_audit.loc[leakage_audit["status"] != "pass"]
failed_feature_contracts = generic_feature_audit.loc[generic_feature_audit["status"] != "pass"]
if not failed_leakage_checks.empty or not failed_feature_contracts.empty:
    display(failed_leakage_checks)
    display(failed_feature_contracts)
    if not leakage_detail.empty:
        display(leakage_detail.loc[leakage_detail["time_violation"] | (leakage_detail["source_end_time"].isna() & leakage_detail["feature_has_value"])] )
    display(same_month_component_detail.loc[
        (~same_month_component_detail["source_bizym_before_target"])
        | (~same_month_component_detail["source_month_ends_before_target_month"])
        | (~same_month_component_detail["component_is_missing_when_source_missing"])
        | (~same_month_component_detail["component_matches_recomputed_history"])
    ])
    raise AssertionError("Point-in-time feature leakage audit failed; downstream metrics are not trustworthy.")
else:
    print("Universal point-in-time feature leakage audit passed.")


## 3. Relationship with Monthly Total

The first chart group looks at two levels:

- Raw single feature: historical same-month workday average quantity vs. actual monthly total.
- Scale-restored view: `same_month_workday_avg_qty_mean × current_month_workdays`, which converts the workday average back to monthly-total scale.

In [ ]:
analysis_monthly = feature_panel.dropna(subset=[FEATURE_COL, TARGET_COL]).copy().sort_values("month_start")


def safe_corr(x: pd.Series, y: pd.Series, method: str) -> tuple[float, float]:
    aligned = pd.concat([x, y], axis=1).dropna()
    if len(aligned) < 3 or aligned.iloc[:, 0].nunique() < 2 or aligned.iloc[:, 1].nunique() < 2:
        return np.nan, np.nan
    if method == "spearman":
        corr, p_value = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    elif method == "pearson":
        corr, p_value = stats.pearsonr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    else:
        raise ValueError(f"Unsupported correlation method: {method}")
    return float(corr), float(p_value)


def prediction_metrics(y_true: pd.Series, y_pred: pd.Series, label: str) -> dict[str, float | str]:
    aligned = pd.concat([y_true.rename("y_true"), y_pred.rename("y_pred")], axis=1).dropna()
    err = aligned["y_pred"] - aligned["y_true"]
    abs_err = err.abs()
    denom = aligned["y_true"].abs().replace(0, np.nan)
    ape = abs_err / denom * 100
    smape = 200 * abs_err / (aligned["y_true"].abs() + aligned["y_pred"].abs()).replace(0, np.nan)
    return {
        "model": label,
        "n": len(aligned),
        "mae": abs_err.mean(),
        "rmse": np.sqrt(np.mean(np.square(err))),
        "mape_pct": ape.mean(),
        "median_ape_pct": ape.median(),
        "p75_ape_pct": ape.quantile(0.75),
        "p90_ape_pct": ape.quantile(0.90),
        "max_ape_pct": ape.max(),
        "smape_pct": smape.mean(),
        "wape_pct": abs_err.sum() / aligned["y_true"].abs().sum() * 100,
        "bias_qty": err.mean(),
        "bias_pct": (err / denom * 100).mean(),
    }


def manual_ols_stats(y_true: pd.Series, x_feature: pd.Series, label: str) -> dict[str, float | str]:
    aligned = pd.concat([y_true.rename("y"), x_feature.rename("x")], axis=1).dropna()
    n = len(aligned)
    x = aligned["x"].to_numpy(dtype=float)
    y = aligned["y"].to_numpy(dtype=float)
    x_mat = np.column_stack([np.ones(n), x])
    beta = np.linalg.lstsq(x_mat, y, rcond=None)[0]
    fitted = x_mat @ beta
    residual = y - fitted
    dof = n - x_mat.shape[1]
    sigma2 = np.sum(residual**2) / dof if dof > 0 else np.nan
    cov = sigma2 * np.linalg.inv(x_mat.T @ x_mat) if dof > 0 else np.full((2, 2), np.nan)
    se = np.sqrt(np.diag(cov))
    t_values = beta / se
    p_values = 2 * stats.t.sf(np.abs(t_values), dof) if dof > 0 else np.full(2, np.nan)
    r2_manual = 1 - np.sum(residual**2) / np.sum((y - y.mean()) ** 2)
    return {
        "model": label,
        "n": n,
        "intercept": beta[0],
        "coef": beta[1],
        "coef_t_stat": t_values[1],
        "coef_p_value": p_values[1],
        "r2": r2_manual,
        "residual_dw": np.nan,
        "ljung_box_p_lag3": np.nan,
        "hac_coef_t_stat": np.nan,
        "hac_coef_p_value": np.nan,
        "covariance_note": "ordinary_ols_manual; install statsmodels for HAC/Newey-West",
    }


def ols_significance_stats(y_true: pd.Series, x_feature: pd.Series, label: str) -> dict[str, float | str]:
    if not STATSMODELS_AVAILABLE:
        return manual_ols_stats(y_true, x_feature, label)

    aligned = pd.concat([y_true.rename("y"), x_feature.rename("x")], axis=1).dropna()
    x_design = sm.add_constant(aligned["x"])
    ols = sm.OLS(aligned["y"], x_design).fit()
    hac_lags = min(3, max(1, len(aligned) // 12))
    ols_hac = sm.OLS(aligned["y"], x_design).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    lb_p = acorr_ljungbox(ols.resid, lags=[min(3, max(1, len(aligned) // 4))], return_df=True)["lb_pvalue"].iloc[0]
    return {
        "model": label,
        "n": len(aligned),
        "intercept": ols.params["const"],
        "coef": ols.params["x"],
        "coef_t_stat": ols.tvalues["x"],
        "coef_p_value": ols.pvalues["x"],
        "r2": ols.rsquared,
        "residual_dw": durbin_watson(ols.resid),
        "ljung_box_p_lag3": lb_p,
        "hac_coef_t_stat": ols_hac.tvalues["x"],
        "hac_coef_p_value": ols_hac.pvalues["x"],
        "covariance_note": f"ordinary_ols_and_hac_maxlags_{hac_lags}",
    }

x = analysis_monthly[[FEATURE_COL]].to_numpy(dtype=float)
y = analysis_monthly[TARGET_COL].to_numpy(dtype=float)
linear_model = LinearRegression().fit(x, y)
analysis_monthly["linear_fit_month_total"] = linear_model.predict(x)
analysis_monthly["linear_fit_error"] = analysis_monthly["linear_fit_month_total"] - analysis_monthly[TARGET_COL]
analysis_monthly["linear_fit_mape_pct"] = (analysis_monthly["linear_fit_error"].abs() / analysis_monthly[TARGET_COL] * 100)

corr_pearson, pearson_p_value = safe_corr(analysis_monthly[FEATURE_COL], analysis_monthly[TARGET_COL], "pearson")
corr_spearman, spearman_p_value = safe_corr(analysis_monthly[FEATURE_COL], analysis_monthly[TARGET_COL], "spearman")
r2 = r2_score(y, analysis_monthly["linear_fit_month_total"])
linear_mae = mean_absolute_error(y, analysis_monthly["linear_fit_month_total"])
linear_mape = analysis_monthly["linear_fit_mape_pct"].mean()
implied_mae = mean_absolute_error(y, analysis_monthly["single_feature_implied_month_total"])
implied_mape = analysis_monthly["single_feature_mape_pct"].mean()
implied_bias = analysis_monthly["single_feature_error_pct"].mean()

association_metrics = pd.DataFrame([
    {"metric": "ts_pearson_corr_feature_actual", "value": corr_pearson, "p_value": pearson_p_value, "note": "time-series association, not cross-sectional IC"},
    {"metric": "ts_spearman_corr_feature_actual", "value": corr_spearman, "p_value": spearman_p_value, "note": "time-series rank association, not cross-sectional Rank IC"},
])

ols_metrics = pd.DataFrame([
    ols_significance_stats(analysis_monthly[TARGET_COL], analysis_monthly[FEATURE_COL], "actual_month_total ~ feature")
])

component_prediction_rows = [
    prediction_metrics(analysis_monthly[TARGET_COL], analysis_monthly["single_feature_implied_month_total"], "3y_mean_feature_implied_total"),
    prediction_metrics(analysis_monthly[TARGET_COL], analysis_monthly["linear_fit_month_total"], "in_sample_single_feature_linear_fit"),
]
for years_back in HISTORY_YEARS:
    component_col = f"same_month_y{years_back}_workday_avg_qty"
    pred_col = f"same_month_y{years_back}_implied_month_total"
    analysis_monthly[pred_col] = analysis_monthly[component_col] * analysis_monthly["month_total_workdays"]
    component_prediction_rows.append(
        prediction_metrics(analysis_monthly[TARGET_COL], analysis_monthly[pred_col], f"y{years_back}_only_implied_total")
    )

prediction_metric_table = pd.DataFrame(component_prediction_rows)
metrics = pd.concat([
    association_metrics[["metric", "value", "p_value", "note"]],
    pd.DataFrame([
        {"metric": "single_feature_linear_fit_r2", "value": r2, "p_value": np.nan, "note": "in-sample sklearn fit"},
        {"metric": "single_feature_linear_fit_mae", "value": linear_mae, "p_value": np.nan, "note": "in-sample sklearn fit"},
        {"metric": "single_feature_linear_fit_mape_pct", "value": linear_mape, "p_value": np.nan, "note": "in-sample sklearn fit"},
        {"metric": "implied_month_total_mae", "value": implied_mae, "p_value": np.nan, "note": "feature multiplied by current-month workdays"},
        {"metric": "implied_month_total_mape_pct", "value": implied_mape, "p_value": np.nan, "note": "feature multiplied by current-month workdays"},
        {"metric": "implied_month_total_bias_pct", "value": implied_bias, "p_value": np.nan, "note": "feature multiplied by current-month workdays"},
    ])
], ignore_index=True)

display(metrics)
display(ols_metrics)
display(prediction_metric_table)


## 3.1 Temporal Robustness and OOS Checks

For a non-cross-sectional monthly series, the main failure modes are look-ahead bias, overfit linear calibration, and regime instability. The checks below therefore use chronological holdout, expanding walk-forward, and rolling/expanding time-series association instead of cross-sectional IC.


In [ ]:
def chronological_holdout_linear_fit(df: pd.DataFrame, holdout_fraction: float = HOLDOUT_FRACTION) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = df.dropna(subset=[FEATURE_COL, TARGET_COL]).sort_values("month_start").copy()
    split_idx = int(np.floor(len(ordered) * (1 - holdout_fraction)))
    split_idx = min(max(split_idx, 3), len(ordered) - 1)
    train = ordered.iloc[:split_idx].copy()
    test = ordered.iloc[split_idx:].copy()
    model = LinearRegression().fit(train[[FEATURE_COL]], train[TARGET_COL])
    train["chronological_linear_pred"] = model.predict(train[[FEATURE_COL]])
    test["chronological_linear_pred"] = model.predict(test[[FEATURE_COL]])
    split_summary = pd.DataFrame([
        {
            "split": "train",
            "n": len(train),
            "first_bizym": train["bizym"].min(),
            "last_bizym": train["bizym"].max(),
        },
        {
            "split": "test",
            "n": len(test),
            "first_bizym": test["bizym"].min(),
            "last_bizym": test["bizym"].max(),
        },
    ])
    metric_rows = [
        prediction_metrics(train[TARGET_COL], train["chronological_linear_pred"], "chronological_train_linear_fit"),
        prediction_metrics(test[TARGET_COL], test["chronological_linear_pred"], "chronological_test_linear_fit"),
        prediction_metrics(test[TARGET_COL], test["single_feature_implied_month_total"], "chronological_test_3y_mean_implied_total"),
    ]
    return split_summary, pd.DataFrame(metric_rows)


def expanding_walk_forward_linear_fit(df: pd.DataFrame, min_train_months: int = MIN_WALK_FORWARD_TRAIN_MONTHS) -> pd.DataFrame:
    ordered = df.dropna(subset=[FEATURE_COL, TARGET_COL]).sort_values("month_start").reset_index(drop=True)
    if len(ordered) <= min_train_months:
        return pd.DataFrame()
    rows = []
    for test_idx in range(min_train_months, len(ordered)):
        train = ordered.iloc[:test_idx]
        test_row = ordered.iloc[[test_idx]].copy()
        model = LinearRegression().fit(train[[FEATURE_COL]], train[TARGET_COL])
        pred = float(model.predict(test_row[[FEATURE_COL]])[0])
        actual = float(test_row[TARGET_COL].iloc[0])
        rows.append({
            "bizym": int(test_row["bizym"].iloc[0]),
            "month_start": test_row["month_start"].iloc[0],
            "train_months": len(train),
            "actual_month_total": actual,
            "walk_forward_linear_pred": pred,
            "walk_forward_error_pct": (pred - actual) / actual * 100,
            "walk_forward_ape_pct": abs(pred - actual) / actual * 100,
            "feature_implied_pred": float(test_row["single_feature_implied_month_total"].iloc[0]),
            "feature_implied_ape_pct": float(test_row["single_feature_mape_pct"].iloc[0]),
        })
    return pd.DataFrame(rows)

holdout_split_summary, holdout_metrics = chronological_holdout_linear_fit(analysis_monthly)
walk_forward_results = expanding_walk_forward_linear_fit(analysis_monthly)

if walk_forward_results.empty:
    walk_forward_metrics = pd.DataFrame([{"model": "expanding_walk_forward_linear_fit", "n": 0, "note": "not enough months"}])
else:
    walk_forward_metrics = pd.DataFrame([
        prediction_metrics(walk_forward_results["actual_month_total"], walk_forward_results["walk_forward_linear_pred"], "expanding_walk_forward_linear_fit"),
        prediction_metrics(walk_forward_results["actual_month_total"], walk_forward_results["feature_implied_pred"], "walk_forward_period_3y_mean_implied_total"),
    ])

rolling_stability = analysis_monthly[["bizym", "month_start", FEATURE_COL, TARGET_COL, "single_feature_mape_pct", "single_feature_error_pct"]].copy()
rolling_stability["rolling_pearson_corr"] = rolling_stability[FEATURE_COL].rolling(
    ROLLING_WINDOW_MONTHS, min_periods=MIN_ROLLING_PERIODS
).corr(rolling_stability[TARGET_COL])
rolling_stability["rolling_spearman_corr"] = rolling_stability[FEATURE_COL].rank().rolling(
    ROLLING_WINDOW_MONTHS, min_periods=MIN_ROLLING_PERIODS
).corr(rolling_stability[TARGET_COL].rank())
rolling_stability["expanding_pearson_corr"] = rolling_stability[FEATURE_COL].expanding(MIN_ROLLING_PERIODS).corr(rolling_stability[TARGET_COL])
rolling_stability["rolling_mean_mape_pct"] = rolling_stability["single_feature_mape_pct"].rolling(
    ROLLING_WINDOW_MONTHS, min_periods=MIN_ROLLING_PERIODS
).mean()

stability_summary = pd.DataFrame([
    {
        "metric": "rolling_pearson_mean",
        "value": rolling_stability["rolling_pearson_corr"].mean(),
        "note": f"{ROLLING_WINDOW_MONTHS}m rolling, min {MIN_ROLLING_PERIODS} months",
    },
    {
        "metric": "rolling_pearson_positive_rate_pct",
        "value": (rolling_stability["rolling_pearson_corr"].dropna() > 0).mean() * 100,
        "note": "time-series stability; not cross-sectional IC positive rate",
    },
    {
        "metric": "rolling_spearman_mean",
        "value": rolling_stability["rolling_spearman_corr"].mean(),
        "note": f"{ROLLING_WINDOW_MONTHS}m rolling rank association",
    },
    {
        "metric": "rolling_spearman_positive_rate_pct",
        "value": (rolling_stability["rolling_spearman_corr"].dropna() > 0).mean() * 100,
        "note": "time-series rank stability; not cross-sectional Rank IC",
    },
    {
        "metric": "rolling_mape_mean_pct",
        "value": rolling_stability["rolling_mean_mape_pct"].mean(),
        "note": f"{ROLLING_WINDOW_MONTHS}m rolling MAPE",
    },
])

display(holdout_split_summary)
display(holdout_metrics)
display(walk_forward_metrics)
display(stability_summary)
display(rolling_stability.tail(12))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(analysis_monthly["month_start"], analysis_monthly[TARGET_COL], marker="o", label="Actual month total")
axes[0].plot(analysis_monthly["month_start"], analysis_monthly["single_feature_implied_month_total"], marker="o", label="Feature implied month total")
axes[0].set_title("Actual Monthly Total vs Single-Feature Implied Monthly Total")
axes[0].set_ylabel("qty")
axes[0].legend()

axes[1].bar(analysis_monthly["month_start"], analysis_monthly["single_feature_error_pct"], width=20)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Single-Feature Implied Monthly Total Error Rate")
axes[1].set_ylabel("error %")
axes[1].set_xlabel("month")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(
    data=analysis_monthly,
    x=FEATURE_COL,
    y=TARGET_COL,
    scatter_kws={"s": 55, "alpha": 0.8},
    line_kws={"color": "#C44E52", "linewidth": 2},
    ax=axes[0],
)
axes[0].set_title("Raw Single Feature vs Actual Monthly Total")
axes[0].set_xlabel("same_month_workday_avg_qty_mean")
axes[0].set_ylabel("actual_month_total")

sns.scatterplot(
    data=analysis_monthly,
    x="single_feature_implied_month_total",
    y=TARGET_COL,
    hue="history_years_available",
    palette="viridis",
    s=70,
    ax=axes[1],
)
min_v = min(analysis_monthly["single_feature_implied_month_total"].min(), analysis_monthly[TARGET_COL].min())
max_v = max(analysis_monthly["single_feature_implied_month_total"].max(), analysis_monthly[TARGET_COL].max())
axes[1].plot([min_v, max_v], [min_v, max_v], color="black", linestyle="--", linewidth=1)
axes[1].set_title("Single-Feature Implied Monthly Total vs Actual Monthly Total")
axes[1].set_xlabel("same_month_workday_avg_qty_mean × current_month_workdays")
axes[1].set_ylabel("actual_month_total")

for _, row in analysis_monthly.nlargest(5, "single_feature_mape_pct").iterrows():
    axes[1].annotate(str(int(row["bizym"])), (row["single_feature_implied_month_total"], row[TARGET_COL]), fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
top_error_months = analysis_monthly.sort_values("single_feature_mape_pct", ascending=False)[[
    "bizym",
    "history_years_available",
    FEATURE_COL,
    "month_total_workdays",
    "single_feature_implied_month_total",
    TARGET_COL,
    "single_feature_error_pct",
    "single_feature_mape_pct",
]].head(12)
display(top_error_months)

## 4. Anchor Impact (WD1 to WD20+)

This feature comes from historical same-month workday averages. Within the same target month, it does not change as the anchor moves from `WD1` to `WD20+`. Therefore, the WD analysis below does not imply that the feature gains more in-month information; it shows how error, bias, and sample coverage behave when this fixed single feature is evaluated across different forecast-anchor slices.

In [ ]:
def wd_bucket(seq: int) -> str:
    return "WD20+" if int(seq) >= 20 else f"WD{int(seq)}"

anchor_rows = []
for _, row in feature_panel.iterrows():
    if pd.isna(row[FEATURE_COL]):
        continue
    for seq in range(1, int(row["month_total_workdays"]) + 1):
        anchor_rows.append({
            "bizym": int(row["bizym"]),
            "month_start": row["month_start"],
            "forecast_workday_seq": seq,
            "wd_bucket": wd_bucket(seq),
            FEATURE_COL: row[FEATURE_COL],
            "month_total_workdays": int(row["month_total_workdays"]),
            TARGET_COL: row[TARGET_COL],
            "single_feature_implied_month_total": row["single_feature_implied_month_total"],
            "single_feature_error_pct": row["single_feature_error_pct"],
            "single_feature_mape_pct": row["single_feature_mape_pct"],
        })

anchor_panel = pd.DataFrame(anchor_rows)
anchor_panel["wd_bucket_order"] = np.where(
    anchor_panel["forecast_workday_seq"] >= 20,
    20,
    anchor_panel["forecast_workday_seq"],
)

wd_summary = (
    anchor_panel.groupby(["wd_bucket_order", "wd_bucket"], as_index=False)
    .agg(
        months=("bizym", "nunique"),
        rows=("bizym", "count"),
        mean_mape_pct=("single_feature_mape_pct", "mean"),
        median_mape_pct=("single_feature_mape_pct", "median"),
        mean_bias_pct=("single_feature_error_pct", "mean"),
        p75_mape_pct=("single_feature_mape_pct", lambda s: s.quantile(0.75)),
    )
    .sort_values("wd_bucket_order")
)

display(wd_summary)

## 4.1 Why WD1-WDN Forecasting Is Hard, and Why This Feature Helps

The core difficulty from `WD1` to `WDN` is information scarcity. Earlier anchors have fewer completed workdays in the current month, so a model has less in-month sales evidence for the final monthly total. Later anchors have fewer unknown remaining workdays, but their evaluation slices also become more concentrated in months with many workdays.

The value of `same_month_workday_avg_qty_mean` is that it is already available before `WD1` and does not depend on current-month observed sales. It compresses historical same-month seasonal demand into a workday-average quantity, which can be multiplied by current-month workdays to return to monthly-total scale. This gives the model a stable, interpretable, pre-anchor baseline signal.

In [ ]:
wd_difficulty = wd_summary.copy()

def completed_workdays_from_bucket(bucket: str) -> int:
    return 19 if bucket == "WD20+" else int(bucket.replace("WD", "")) - 1

wd_difficulty["completed_workdays_before_forecast"] = wd_difficulty["wd_bucket"].map(completed_workdays_from_bucket)
mean_workdays_by_wd = (
    anchor_panel.groupby(["wd_bucket_order", "wd_bucket"], as_index=False)["month_total_workdays"]
    .mean()
    .rename(columns={"month_total_workdays": "mean_month_workdays_in_slice"})
)
wd_difficulty = wd_difficulty.merge(mean_workdays_by_wd, on=["wd_bucket_order", "wd_bucket"], how="left")
wd_difficulty["observed_workday_share_before_forecast_pct"] = (
    wd_difficulty["completed_workdays_before_forecast"]
    / wd_difficulty["mean_month_workdays_in_slice"].replace(0, np.nan)
    * 100
).clip(upper=100)
wd_difficulty["unknown_workday_share_at_forecast_pct"] = (
    100 - wd_difficulty["observed_workday_share_before_forecast_pct"]
)

def corr_pair_for_group(d: pd.DataFrame, method: str) -> float:
    if d[FEATURE_COL].nunique() <= 1 or d[TARGET_COL].nunique() <= 1 or len(d) < 3:
        return np.nan
    return safe_corr(d[FEATURE_COL], d[TARGET_COL], method)[0]

feature_corr_by_wd = (
    anchor_panel.groupby(["wd_bucket_order", "wd_bucket"])
    .apply(lambda d: corr_pair_for_group(d, "pearson"), include_groups=False)
    .reset_index(name="feature_actual_ts_pearson_corr")
)
feature_rank_corr_by_wd = (
    anchor_panel.groupby(["wd_bucket_order", "wd_bucket"])
    .apply(lambda d: corr_pair_for_group(d, "spearman"), include_groups=False)
    .reset_index(name="feature_actual_ts_spearman_corr")
)
wd_difficulty = wd_difficulty.merge(feature_corr_by_wd, on=["wd_bucket_order", "wd_bucket"], how="left")
wd_difficulty = wd_difficulty.merge(feature_rank_corr_by_wd, on=["wd_bucket_order", "wd_bucket"], how="left")
wd_difficulty["feature_known_before_wd1"] = True

# Note: WD rows repeat the same monthly feature across anchors. These are anchor-slice diagnostics, not cross-sectional IC.
display(wd_difficulty[[
    "wd_bucket",
    "months",
    "rows",
    "completed_workdays_before_forecast",
    "observed_workday_share_before_forecast_pct",
    "unknown_workday_share_at_forecast_pct",
    "feature_actual_ts_pearson_corr",
    "feature_actual_ts_spearman_corr",
    "mean_mape_pct",
    "mean_bias_pct",
    "feature_known_before_wd1",
]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(
    data=wd_difficulty,
    x="wd_bucket",
    y="unknown_workday_share_at_forecast_pct",
    marker="o",
    color="#C44E52",
    ax=axes[0],
)
axes[0].set_title("Forecast Difficulty: Earlier Anchors Have More Unknown Workdays")
axes[0].set_xlabel("forecast workday bucket")
axes[0].set_ylabel("unknown workday share %")
axes[0].tick_params(axis="x", rotation=45)

sns.lineplot(
    data=wd_difficulty,
    x="wd_bucket",
    y="feature_actual_ts_pearson_corr",
    marker="o",
    label="TS Pearson corr(feature, actual total)",
    ax=axes[1],
)
ax2 = axes[1].twinx()
sns.lineplot(
    data=wd_difficulty,
    x="wd_bucket",
    y="mean_mape_pct",
    marker="o",
    color="#55A868",
    label="Feature implied MAPE",
    ax=ax2,
)
axes[1].set_title("Explanatory Value: Pre-WD Historical Seasonal Intensity")
axes[1].set_xlabel("forecast workday bucket")
axes[1].set_ylabel("correlation")
ax2.set_ylabel("MAPE %")
axes[1].tick_params(axis="x", rotation=45)
lines, labels = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines + lines2, labels + labels2, loc="best")
ax2.legend_.remove() if ax2.legend_ else None

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

sns.lineplot(data=wd_summary, x="wd_bucket", y="mean_mape_pct", marker="o", ax=axes[0], label="Mean MAPE")
sns.lineplot(data=wd_summary, x="wd_bucket", y="median_mape_pct", marker="o", ax=axes[0], label="Median MAPE")
axes[0].set_title("Single-Feature Implied Monthly Total MAPE by Forecast WD")
axes[0].set_ylabel("MAPE %")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=wd_summary, x="wd_bucket", y="mean_bias_pct", ax=axes[1], color="#4C72B0")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Single-Feature Implied Monthly Total Bias by Forecast WD")
axes[1].set_ylabel("Bias %")
axes[1].set_xlabel("forecast workday bucket")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13, 5))
sns.boxplot(
    data=anchor_panel.sort_values("wd_bucket_order"),
    x="wd_bucket",
    y="single_feature_error_pct",
    color="#55A868",
)
plt.axhline(0, color="black", linewidth=1)
plt.title("Single-Feature Error Rate Distribution by Forecast WD")
plt.xlabel("forecast workday bucket")
plt.ylabel("error %")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = anchor_panel.pivot_table(
    index="bizym",
    columns="wd_bucket",
    values="single_feature_mape_pct",
    aggfunc="mean",
)
ordered_cols = [f"WD{i}" for i in range(1, 20) if f"WD{i}" in heatmap_data.columns]
if "WD20+" in heatmap_data.columns:
    ordered_cols.append("WD20+")
heatmap_data = heatmap_data[ordered_cols]

plt.figure(figsize=(13, max(6, len(heatmap_data) * 0.22)))
sns.heatmap(heatmap_data, cmap="YlOrRd", linewidths=0.2, linecolor="white", cbar_kws={"label": "MAPE %"})
plt.title("WD x Month: Single-Feature Implied Monthly Total MAPE")
plt.xlabel("forecast workday bucket")
plt.ylabel("bizym")
plt.tight_layout()
plt.show()

## 5. Time Slice Stability

These slices are valid for the current aggregate time-series grain. They test whether the feature is robust by calendar year, calendar month, and history coverage. Category/SKU/store slices are intentionally absent because the data in this notebook is not a panel.


In [ ]:
def slice_summary(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    for group_key, g in df.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = {col: value for col, value in zip(group_cols, group_key)}
        row.update({
            "months": len(g),
            "feature_missing_rate_pct": g[FEATURE_COL].isna().mean() * 100,
            "ts_pearson_corr": safe_corr(g[FEATURE_COL], g[TARGET_COL], "pearson")[0] if len(g.dropna(subset=[FEATURE_COL, TARGET_COL])) >= 3 else np.nan,
            "ts_spearman_corr": safe_corr(g[FEATURE_COL], g[TARGET_COL], "spearman")[0] if len(g.dropna(subset=[FEATURE_COL, TARGET_COL])) >= 3 else np.nan,
            "mean_mape_pct": g["single_feature_mape_pct"].mean(),
            "median_mape_pct": g["single_feature_mape_pct"].median(),
            "mean_bias_pct": g["single_feature_error_pct"].mean(),
            "p90_mape_pct": g["single_feature_mape_pct"].quantile(0.90),
        })
        rows.append(row)
    return pd.DataFrame(rows)

year_stability = slice_summary(analysis_monthly, ["year"]).sort_values("year")
month_of_year_stability = slice_summary(analysis_monthly, ["month_num"]).sort_values("month_num")
history_coverage_stability = slice_summary(analysis_monthly, ["history_years_available"]).sort_values("history_years_available")

print("Year stability:")
display(year_stability)
print("Month-of-year stability:")
display(month_of_year_stability)
print("History coverage stability:")
display(history_coverage_stability)


## 5. Auto-Generated Summary

The next cell generates key takeaways for this single-feature analysis.

In [ ]:
best_wd = wd_summary.loc[wd_summary["mean_mape_pct"].idxmin()]
worst_wd = wd_summary.loc[wd_summary["mean_mape_pct"].idxmax()]
coef_t_stat = ols_metrics["coef_t_stat"].iloc[0]
coef_p_value = ols_metrics["coef_p_value"].iloc[0]
hac_t_stat = ols_metrics["hac_coef_t_stat"].iloc[0]
hac_p_value = ols_metrics["hac_coef_p_value"].iloc[0]
hac_text = (
    f"HAC t={hac_t_stat:.2f}, p={hac_p_value:.4f}"
    if pd.notna(hac_t_stat)
    else "HAC unavailable because statsmodels is not installed"
)
leakage_status_text = "passed" if (leakage_audit["status"] == "pass").all() else "failed"
contract_count = len(analysis_feature_contracts)

summary_lines = [
    f"1. This is a non-cross-sectional time-series review: no cross-sectional IC is computed because each month has one aggregate target.",
    f"2. Universal point-in-time leakage audit {leakage_status_text}: {contract_count} feature timing contract(s) registered; source_end_time is checked against prediction_time, forbidden label/evaluation/future fields are blocked, and same-month history has an additional recomputation check.",
    f"3. The single feature {FEATURE_COL} has time-series Pearson correlation {corr_pearson:.3f} (p={pearson_p_value:.4f}) and Spearman correlation {corr_spearman:.3f} (p={spearman_p_value:.4f}) with actual_month_total.",
    f"4. OLS coefficient t={coef_t_stat:.2f}, p={coef_p_value:.4f}; {hac_text}.",
    f"5. A one-variable in-sample linear fit explains monthly total with R2={r2:.3f} and MAPE={linear_mape:.2f}%.",
    f"6. After restoring the feature to monthly-total scale using current-month workdays, MAPE={implied_mape:.2f}% and average Bias={implied_bias:.2f}%.",
    f"7. Across WD buckets, the lowest mean MAPE is {best_wd['wd_bucket']} ({best_wd['mean_mape_pct']:.2f}%), and the highest is {worst_wd['wd_bucket']} ({worst_wd['mean_mape_pct']:.2f}%).",
    "8. Within the same target month, this feature does not change as the anchor advances. Differences across WD1-WD20+ mainly come from month coverage in each WD slice and aggregation in WD20+, not from new in-month information entering the feature.",
]

for line in summary_lines:
    print(line)


In [ ]:
# Sanity checks for the single-feature analysis contract.
assert FEATURE_COL == "same_month_workday_avg_qty_mean"
assert analysis_feature_cols == [FEATURE_COL]
assert set(history_cols) == {
    "same_month_y1_workday_avg_qty",
    "same_month_y2_workday_avg_qty",
    "same_month_y3_workday_avg_qty",
}
assert anchor_panel[FEATURE_COL].notna().all()
assert anchor_panel[TARGET_COL].notna().all()
assert anchor_panel["wd_bucket"].str.match(r"^WD(\d+|20\+)$").all()
assert "feature_actual_ts_pearson_corr" in wd_difficulty.columns
assert "feature_actual_ts_spearman_corr" in wd_difficulty.columns
assert "coef_t_stat" in ols_metrics.columns
assert leakage_contract.loc[leakage_contract["check"] == "cross_sectional_ic_applicable", "status"].iloc[0] == "not_applicable"
assert set(analysis_feature_cols).issubset(analysis_feature_contracts.keys())
assert (generic_feature_audit["status"] == "pass").all()
assert (leakage_audit["status"] == "pass").all()
assert not leakage_detail.empty
assert leakage_detail["time_violation"].sum() == 0
assert same_month_component_detail["source_bizym_before_target"].all()
assert same_month_component_detail["source_month_ends_before_target_month"].all()
assert same_month_component_detail["component_matches_recomputed_history"].all()
assert not any("cross_sectional_ic" in metric for metric in metrics["metric"].astype(str))
print("All non-cross-sectional single-feature analysis and universal point-in-time leakage checks passed.")
